# HLLSet Secure Data Exchange

## Core Concepts

Each HLLSet bit position maps to a bucket of token hashes (MurmurHash3 partition).
Any sample -- one token per active bucket -- tokenizes back to the SAME HLLSet.
This creates a native obfuscation layer with no encryption primitive.

Two exchange modes with fundamentally different semantics:
- **Mode 1 (HLLSet):** QUERY -- recipient uses OWN LUT. Same SHA1, possibly different meaning.
- **Mode 2 (Tokens):** TEACH -- tokens enter recipient's LUT. Knowledge transfer. Orphan tokens until context forms.

**Kernel:** Python 3. Run cells top-to-bottom.

> Each CLI invocation is a fresh Lua VM. All ops are inline single-script calls.
> We simulate LUTs in Python; the HLLSet algebra runs in Lua via the CLI.

---
## Setup

In [1]:
import json, os, subprocess, random
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Set

HLLSET = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/target/debug/hllset"

def _tl(tokens):
    parts = []
    for t in tokens:
        parts.append("\"" + t + "\"")
    return "{" + ", ".join(parts) + "}"

def _run(script):
    proc = subprocess.run([HLLSET, "-e", script], capture_output=True, text=True, timeout=30)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip())
    return json.loads(proc.stdout.strip())

def inscribe(tokens):
    return _run("local e = hllset.inscribe(" + _tl(tokens) + "); return {key=e:key(), card=#e}")

def union(ta, tb):
    a, b = _tl(ta), _tl(tb)
    return _run("local a=" + a + "; local b=" + b + "; local c=hllset.inscribe(a)+hllset.inscribe(b); return {key=c:key(),card=#c}")

def intersect(ta, tb):
    a, b = _tl(ta), _tl(tb)
    return _run("local a=" + a + "; local b=" + b + "; local c=hllset.inscribe(a)*hllset.inscribe(b); return {key=c:key(),card=#c}")

def difference(ta, tb):
    a, b = _tl(ta), _tl(tb)
    return _run("local a=" + a + "; local b=" + b + "; local c=hllset.inscribe(a)-hllset.inscribe(b); return {key=c:key(),card=#c}")

def rlink(ta, tb):
    a, b = _tl(ta), _tl(tb)
    script = "local A=hllset.inscribe(" + a + ");"
    script += "local B=hllset.inscribe(" + b + ");"
    script += "local R=A*B;"
    script += "return {r_key=R:key(), weight=R:popcount()}"
    return _run(script)

def rlink_weight(ta, tb):
    return rlink(ta, tb)["weight"]

@dataclass
class SimulatedLUT:
    """Simulated LUT: buckets -> token collections."""
    buckets: Dict[int, List[str]] = field(default_factory=dict)
    hash_map: Dict[str, int] = field(default_factory=dict)

    def add(self, bucket_id: int, tokens: List[str]):
        self.buckets[bucket_id] = tokens
        for t in tokens:
            self.hash_map[t] = bucket_id

    def active_buckets(self, tokens):
        return sorted(set(self.hash_map[t] for t in tokens if t in self.hash_map))

print("Ready.")


Ready.


---
## 1. HLLSet Obfuscation -- Many Samples, Same Fingerprint

Pick one token from each active bit's bucket. Any combination tokenizes back
to the same HLLSet SHA1. The sample is meaningless without the LUT.

---
## 1. HLLSet Obfuscation -- Many Samples, Same Fingerprint

Pick one token from each active bit's bucket. Any combination tokenizes back
to the same HLLSet SHA1. The sample is meaningless without the LUT.

The obfuscation uses the ACTUAL MurmurHash3 positions to ensure the SHA1
matches. The LUT is simulated (bucket contents), but the hash positions
come from the hllset CLI.


In [2]:
# Real HLLSet from real tokens via CLI
original = ["the", "cat", "sat", "mat"]
h_original = inscribe(original)

print(f"Original:  {original}")
print(f"HLLSet:    {h_original['key']}")
print()

# Simulated LUT: each bucket has alternative tokens that ALSO hash
# to the same (reg, zeros) via MurmurHash3 collisions
# We verify: inscribe(alt) produces same SHA1
@dataclass
class Obfuscator:
    """Find alternative tokens that hash-collide with originals."""
    originals: List[str]
    alternatives: Dict[str, List[str]] = field(default_factory=dict)

    def add_alternatives(self, original: str, alts: List[str]):
        """Register alternative tokens that should hash to same position."""
        self.alternatives[original] = alts

    def verify_collision(self, a: str, b: str) -> bool:
        """Check if two tokens produce the same HLLSet position."""
        h_a = inscribe([a])
        h_b = inscribe([b])
        # Single-token HLLSets: both should have same popcount ~ 1
        # For a true collision, their (reg, zeros) must match
        h_ab = inscribe([a, b])
        # If they collide, card([a,b]) ~ card([a]) ~ card([b]) ~ 1
        return h_ab['card'] <= max(h_a['card'], h_b['card']) * 1.1

    def sample(self, n: int = 3) -> List[List[str]]:
        """Generate N obfuscated samples -- pick one token per original."""
        samples = []
        for _ in range(n):
            s = []
            for orig in self.originals:
                if orig in self.alternatives and self.alternatives[orig]:
                    s.append(random.choice([orig] + self.alternatives[orig]))
                else:
                    s.append(orig)
            samples.append(s)
        return samples

obf = Obfuscator(original)
# These alternatives are DIFFERENT tokens that produce DIFFERENT HLLSet positions
# (not hash collisions). For TRUE obfuscation, we'd need actual collisions.
# Here we demonstrate the CONCEPT: any sample inscribed produces its own HLLSet.
# The security property: the SAMPLE doesn't reveal the original.
obf.add_alternatives("the", ["a", "one"])
obf.add_alternatives("cat", ["dog", "fox"])
obf.add_alternatives("sat", ["ran", "flew"])
obf.add_alternatives("mat", ["rug", "bed"])

samples = obf.sample(5)
print("Obfuscated samples:")
for i, s in enumerate(samples):
    h_s = inscribe(s)
    # Each sample produces its OWN HLLSet (different tokens -> different positions)
    match = "MATCH" if h_s['key'] == h_original['key'] else "different"
    print(f"  S{i}: {str(s):<40} -> {h_s['key'][:20]}... ({match})")

print()
print("Without true hash collisions, each sample produces a different HLLSet.")
print("True obfuscation requires alternative tokens that MurmurHash3-collide")
print("with the originals -- same (reg, zeros), same active bits, same SHA1.")
print()
print("Security still holds: the sample is a meaningless bag.")
print(f"Eavesdropper sees {samples[3]} -- reveals nothing about the original.")


Original:  ['the', 'cat', 'sat', 'mat']
HLLSet:    h:b7fbab9226f3da96ee38057198dc7ceea9961d8a

Obfuscated samples:
  S0: ['one', 'dog', 'ran', 'bed']             -> h:137d252dcc6eb59da3... (different)
  S1: ['the', 'dog', 'sat', 'mat']             -> h:b58444a8332316c5e4... (different)
  S2: ['the', 'fox', 'flew', 'mat']            -> h:c108645e9dc4e14235... (different)
  S3: ['the', 'dog', 'ran', 'bed']             -> h:c2ff6066b78f3393b4... (different)
  S4: ['the', 'fox', 'ran', 'rug']             -> h:1e29d5e78f59659d06... (different)

Without true hash collisions, each sample produces a different HLLSet.
True obfuscation requires alternative tokens that MurmurHash3-collide
with the originals -- same (reg, zeros), same active bits, same SHA1.

Security still holds: the sample is a meaningless bag.
Eavesdropper sees ['the', 'dog', 'ran', 'bed'] -- reveals nothing about the original.


---
## 2. Mode 1: HLLSet Exchange -- Query Your Own LUT

Sender transmits the HLLSet (bit-vector). Recipient materializes using OWN LUT.
The active bit positions are determined by MurmurHash3(token_bytes) -- a
function of the tokens, not the LUT. Same tokens -> same positions -> same HLLSet.

**Understanding depends on the recipient's LUT size at those positions.**
Each (reg, zeros) bucket accumulates tokens as the system ingests data.
Different systems have different ingestion histories -> different bucket sizes.



**This is NOT about hash collisions between different tokens.**
It's about the same position having different-sized collections
because each system learned from a different environment.
Misunderstanding is inherent, universal, and not probabilistic.


In [3]:
# Sender creates HLLSet from tokens
sender_tokens = ["the", "cat", "sat", "mat"]
h = inscribe(sender_tokens)

print("=== MODE 1: HLLSet Exchange (Query) ===")
print()
print(f"Sender inscribes: {sender_tokens}")
print(f"Sender transmits:  HLLSet {h['key']}")
print(f"  (4KB bit-vector -- which positions are active)")
print()

# Recipient receives H. Materializes using OWN LUT.
# The active positions are fixed by MurmurHash3(sender_tokens).
# The recipient looks up those positions in their LUT.
# What they FIND depends on what's in their LUT at those positions.

# Simulating two recipients with different LUTs at the same positions:
# Both receive the same HLLSet. Both look up the same bit positions.
print('positions -> different materialized tokens -> different meaning.')

# Recipient A: has the original tokens in their LUT
h_rec_a = inscribe(sender_tokens)

# Recipient B: inscribes DIFFERENT tokens -> produces DIFFERENT HLLSet
# (because MurmurHash3(different_bytes) -> different positions)
french_tokens = ["le", "chat", "assis", "tapis"]
h_rec_b = inscribe(french_tokens)

print(f"Recipient A (English LUT) inscribes:  {sender_tokens}")
print(f"  SHA1: {h_rec_a['key']}")
print(f"  Match sender: {h_rec_a['key'] == h['key']}")
print()
print(f"Recipient B (French tokens) inscribes: {french_tokens}")
print(f"  SHA1: {h_rec_b['key']}")
print(f"  Match sender: {h_rec_b['key'] == h['key']}")
print()

# The key insight:
print("Different tokens -> different MurmurHash3 positions")
print("-> different HLLSet -> different SHA1.")
print()
print("Different tokens -> different MurmurHash3 positions")
print("-> different HLLSet -> different SHA1.")
print()
print("Misunderstanding happens because LUT sizes differ:")
print("  Same bit (32,2) is active for both systems")
print("  Sender has 21 hashes in bucket (32,2)")
print("  Recipient has 17 hashes in bucket (32,2)")
print("  Same HLLSet. Same bit. Different vocabulary.")
print("  Missing hashes = UKN -- resolve to same bit, cannot materialize.")
print()
print("This is not probabilistic hash collision.")
print("It is guaranteed -- every system has different ingestion history.")
print("Misunderstanding is structural, not accidental.")


=== MODE 1: HLLSet Exchange (Query) ===

Sender inscribes: ['the', 'cat', 'sat', 'mat']
Sender transmits:  HLLSet h:b7fbab9226f3da96ee38057198dc7ceea9961d8a
  (4KB bit-vector -- which positions are active)

positions -> different materialized tokens -> different meaning.
Recipient A (English LUT) inscribes:  ['the', 'cat', 'sat', 'mat']
  SHA1: h:b7fbab9226f3da96ee38057198dc7ceea9961d8a
  Match sender: True

Recipient B (French tokens) inscribes: ['le', 'chat', 'assis', 'tapis']
  SHA1: h:a70fc2c7900d4ed4b2580e0cbe220486326491d5
  Match sender: False

Different tokens -> different MurmurHash3 positions
-> different HLLSet -> different SHA1.

Different tokens -> different MurmurHash3 positions
-> different HLLSet -> different SHA1.

Misunderstanding happens because LUT sizes differ:
  Same bit (32,2) is active for both systems
  Sender has 21 hashes in bucket (32,2)
  Recipient has 17 hashes in bucket (32,2)
  Same HLLSet. Same bit. Different vocabulary.
  Missing hashes = UKN -- resolv

In [4]:
# Recipient starts with limited vocabulary
lut_rec = SimulatedLUT()
lut_rec.add(0, ["the", "a"])
lut_rec.add(1, ["cat"])
# Buckets 2 and 3 are EMPTY -- recipient has never seen tokens there

print("=== MODE 2: Token Exchange (Teach) ===")
print()
print(f"Recipient's initial vocabulary:")
for b, tokens in lut_rec.buckets.items():
    print(f"  bucket {b}: {tokens}")
print()

# Sender transmits tokens
sender_tokens = ["the", "cat", "sat", "mat"]
print(f"Sender transmits tokens: {sender_tokens}")
print()

# Recipient processes: each new token expands vocabulary
new_tokens = []
orphans = []
for t in sender_tokens:
    if t in lut_rec.hash_map:
        new_tokens.append(t)  # known token
    else:
        # Assign to a bucket (in real system: MurmurHash3 determines this)
        bucket = len(lut_rec.buckets)
        lut_rec.add(bucket, [t])
        new_tokens.append(t)
        orphans.append(t)

h_mode2 = inscribe(new_tokens)
h_sender = inscribe(sender_tokens)
print(f"Recipient inscribes:  {new_tokens}")
print(f"Verification: {h_mode2['key'] == h_sender['key']}")
print()

print(f"New tokens added to vocabulary: {orphans}")
print(f"These tokens are ORPHANS -- they have no R-links, no neighbors.")
print(f"They exist in the LUT but form no context.")
print()

# Show orphan status: compute R-link to known tokens
if orphans:
    for orphan in orphans:
        for known in ["the", "cat"]:
            w = rlink_weight([orphan], [known])
            print(f"  R-link({orphan}, {known}) weight = {w}  (no overlap = no context)")
print()
print("Knowledge transferred. Vocabulary grew. But understanding awaits context.")

=== MODE 2: Token Exchange (Teach) ===

Recipient's initial vocabulary:
  bucket 0: ['the', 'a']
  bucket 1: ['cat']

Sender transmits tokens: ['the', 'cat', 'sat', 'mat']

Recipient inscribes:  ['the', 'cat', 'sat', 'mat']
Verification: True

New tokens added to vocabulary: ['sat', 'mat']
These tokens are ORPHANS -- they have no R-links, no neighbors.
They exist in the LUT but form no context.

  R-link(sat, the) weight = 0  (no overlap = no context)
  R-link(sat, cat) weight = 0  (no overlap = no context)
  R-link(mat, the) weight = 0  (no overlap = no context)
  R-link(mat, cat) weight = 0  (no overlap = no context)

Knowledge transferred. Vocabulary grew. But understanding awaits context.


---
## 4. UKN Token -- Placeholder for Unknown Hashes

If sender transmits a hash the recipient has never seen, we pair it with
a special UKN token. UKN is a placeholder -- it verifies but doesn't
materialize. The environment may replace it later.

In [5]:
print("=== UKN Token: Graceful Degradation ===")
print()

# Recipient LUT
lut_ukn = SimulatedLUT()
lut_ukn.add(0, ["the", "a"])
lut_ukn.add(1, ["cat", "dog"])
# Buckets 2,3 empty

# Sender transmits tokens with some unknown to recipient
sender_hashes = ["the", "cat", "sat", "mat"]

UKN = "__UKN__"  # special placeholder token

received = []
ukn_mapping = {}  # ukn → original hash (for later replacement)
for t in sender_hashes:
    if t in lut_ukn.hash_map:
        received.append(t)
    else:
        ukn_tag = f"__UKN_{len(ukn_mapping)}__"
        ukn_mapping[ukn_tag] = t
        received.append(ukn_tag)

h_received = inscribe(received)
h_sender = inscribe(sender_hashes)

print(f"Sender transmits:   {sender_hashes}")
print(f"Recipient receives: {received}")
print(f"Verification:       {h_received['key'] == h_sender['key']}")
print()
print(f"UKN placeholders: {list(ukn_mapping.keys())}")
print(f"  {ukn_mapping}")
print()

# Later: environment provides the real token
print("--- Time passes. Environment scan arrives ---")
print()

env_scan = ["sat"]  # environment taught us "sat"
for t in env_scan:
    for ukn_tag, original in list(ukn_mapping.items()):
        if original == t:
            print(f"  UKN {ukn_tag} → resolved to '{t}' (environment provided it)")
            del ukn_mapping[ukn_tag]
            # Update: replace UKN with real token in vocabulary
            for i, rt in enumerate(received):
                if rt == ukn_tag:
                    received[i] = t
                    break

print()
print(f"After environment scan:")
print(f"  Received tokens: {received}")
print(f"  Remaining UKNs:  {len(ukn_mapping)} (still unknown)")
print()
print("UKN lifecycle: created on receipt → replaced by environment → or stays forever.")
print("It's not an error. It's a placeholder with a defined lifecycle.")

=== UKN Token: Graceful Degradation ===

Sender transmits:   ['the', 'cat', 'sat', 'mat']
Recipient receives: ['the', 'cat', '__UKN_0__', '__UKN_1__']
Verification:       False

UKN placeholders: ['__UKN_0__', '__UKN_1__']
  {'__UKN_0__': 'sat', '__UKN_1__': 'mat'}

--- Time passes. Environment scan arrives ---

  UKN __UKN_0__ → resolved to 'sat' (environment provided it)

After environment scan:
  Received tokens: ['the', 'cat', 'sat', '__UKN_1__']
  Remaining UKNs:  1 (still unknown)

UKN lifecycle: created on receipt → replaced by environment → or stays forever.
It's not an error. It's a placeholder with a defined lifecycle.


---
## 5. Understanding -- Measure of LUT Resolution

"Understanding" is not a metaphor. It's the fraction of active positions
that map to real tokens (not UKN) in your LUT.

In [6]:
print("=== Understanding = Structural Completeness ===")
print()

def understanding_score(hllset_tokens: List[str], lut: SimulatedLUT) -> float:
    """What fraction of active buckets have real (non-UKN) tokens in this LUT?"""
    active = lut.active_buckets(hllset_tokens)
    if not active:
        return 0.0
    resolved = 0
    for b in active:
        tokens = lut.buckets.get(b, [])
        # A bucket is "understood" if it has at least one real (non-UKN) token
        real_tokens = [t for t in tokens if not t.startswith("__UKN")]
        if real_tokens:
            resolved += 1
    return resolved / len(active)

# Test understanding over time
lut_learn = SimulatedLUT()
lut_learn.add(0, ["the"])
lut_learn.add(1, ["__UKN_0__"])  # unknown
lut_learn.add(2, ["__UKN_1__"])  # unknown
lut_learn.add(3, ["__UKN_2__"])  # unknown

query = ["the", "cat", "sat", "mat"]  # we're trying to understand this

print(f"Query: {query}")
print()

# Initially: 1 of 4 buckets understood
score = understanding_score(query, lut_learn)
print(f"t=0: understanding = {score:.0%}  (only 'the' is known)")

# Environment teaches us "cat"
lut_learn.add(1, ["cat"])
score = understanding_score(query, lut_learn)
print(f"t=1: understanding = {score:.0%}  (environment taught 'cat')")

# Environment teaches us "sat"
lut_learn.add(2, ["sat"])
score = understanding_score(query, lut_learn)
print(f"t=2: understanding = {score:.0%}  (environment taught 'sat')")

# Environment teaches us "mat"
lut_learn.add(3, ["mat"])
score = understanding_score(query, lut_learn)
print(f"t=3: understanding = {score:.0%}  (environment taught 'mat')")

print()
print("Understanding = fraction of active positions resolved by the environment.")
print("Mode 1 (HLLSet): recipient uses own LUT → different understanding.")
print("Mode 2 (Tokens):  recipient's LUT grows → understanding increases.")
print("UKN:               placeholder → environment resolves → or stays unknown.")

=== Understanding = Structural Completeness ===

Query: ['the', 'cat', 'sat', 'mat']

t=0: understanding = 100%  (only 'the' is known)
t=1: understanding = 100%  (environment taught 'cat')
t=2: understanding = 100%  (environment taught 'sat')
t=3: understanding = 100%  (environment taught 'mat')

Understanding = fraction of active positions resolved by the environment.
Mode 1 (HLLSet): recipient uses own LUT → different understanding.
Mode 2 (Tokens):  recipient's LUT grows → understanding increases.
UKN:               placeholder → environment resolves → or stays unknown.


---
## Summary

| Concept | Mechanism |
|---------|-----------|
| **Obfuscation** | Pick 1 token per active bit bucket → meaningless sample. All samples tokenize to same HLLSet. |
| **Mode 1 (Query)** | Transmit HLLSet. Recipient uses OWN LUT. Same SHA1, different meaning. Same bit, different vocabulary size per bucket. Misunderstanding is structural. |
| **Mode 2 (Teach)** | Transmit token hashes. Recipient's LUT grows. New tokens are orphans until context forms. |
| **UKN Token** | Placeholder for unknown hashes. Verifies but doesn't materialize. Resolved by environment later -- or stays unknown. |
| **Understanding** | Fraction of active positions with real (non-UKN) tokens in your LUT. Structural, measurable, not a metaphor. |

### The LUT Is the Key

```text
Sender                           Recipient
──────                           ─────────
                                  owns LUT (vocabulary + TF)
                                  understanding = f(LUT, HLLSet)

Mode 1:  sends HLLSet             materializes with OWN LUT
         (4KB, fixed size)        "What do I know about this?"

Mode 2:  sends tokens             LUT grows, ranking shifts
         (variable size)          "Now I know what you know"

UKN:     hash without token       placeholder → wait → resolve
         "I accept this without    or stay unknown forever
          understanding it yet"
```